# Autonomous Fleet vs. Traditional Fleet TCO Calculator
**Total Cost of Ownership per kilometer: robotaxis vs. human-driven ride-hailing in high-density markets (Dubai & Singapore presets).**

> All numbers in this notebook are **illustrative modelling assumptions**, not market quotes.
> Calibrate with local data before any commercial use.

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tco_model import (MarketParams, dubai_params, singapore_params,
                       cost_breakdown, breakeven_utilization,
                       breakeven_driver_wage, sensitivity,
                       _robotaxi_total_at_util, _human_total_at_util)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

dubai, sg = dubai_params(), singapore_params()
print("Model loaded. Markets:", dubai.name, "&", sg.name)

Model loaded. Markets: Dubai & Singapore


/home/hatch/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## 1. Setup & assumptions
Every input lives in one visible table per market. All values are illustrative.

In [2]:
rows = [
    ("Base EV price ($)", dubai.base_ev_price, sg.base_ev_price),
    ("Autonomy stack premium ($, robotaxi)", dubai.autonomy_stack_premium, sg.autonomy_stack_premium),
    ("Registration / COE premium ($)", dubai.coe_premium, sg.coe_premium),
    ("Robotaxi utilization (km/day)", dubai.km_per_day_robotaxi, sg.km_per_day_robotaxi),
    ("Human-driven utilization (km/day)", dubai.km_per_day_human, sg.km_per_day_human),
    ("Energy cost ($/kWh)", dubai.energy_cost_per_kwh, sg.energy_cost_per_kwh),
    ("A/C + heat energy penalty", f"{dubai.ac_heat_penalty:.0%}", f"{sg.ac_heat_penalty:.0%}"),
    ("Driver wage ($/hr)", dubai.driver_wage_per_hour, sg.driver_wage_per_hour),
    ("Insurance ($/yr)", dubai.insurance_per_year, sg.insurance_per_year),
    ("Fleet ops ($/veh/day, robotaxi)", dubai.fleet_ops_per_vehicle_day, sg.fleet_ops_per_vehicle_day),
    ("Infrastructure capex ($/veh, robotaxi)", dubai.infra_capex_per_vehicle, sg.infra_capex_per_vehicle),
    ("Battery pack cost ($)", dubai.pack_cost, sg.pack_cost),
    ("Vehicle lifetime (yrs)", dubai.lifetime_years, sg.lifetime_years),
    ("Discount rate", f"{dubai.discount_rate:.0%}", f"{sg.discount_rate:.0%}"),
]
assumptions = pd.DataFrame(rows, columns=["Assumption", "Dubai", "Singapore"]).set_index("Assumption")
assumptions

,Dubai,Singapore
Assumption,,
Base EV price ($),35000.0,35000.0
"Autonomy stack premium ($, robotaxi)",25000.0,25000.0
Registration / COE premium ($),0.0,40000.0
Robotaxi utilization (km/day),320.0,240.0
Human-driven utilization (km/day),280.0,220.0
Energy cost ($/kWh),0.08,0.28
A/C + heat energy penalty,15%,10%
Driver wage ($/hr),9.0,16.0
Insurance ($/yr),1500.0,2500.0


## 2. Model walkthrough
Per-km cost = amortized capex (capital-recovery factor) + energy + battery-degradation reserve
+ maintenance + insurance + fleet ops (robotaxi) + infrastructure (robotaxi) + driver wages (human-driven).

Battery reserve: capacity fades linearly per 1,000 km (heat-adjusted); each time capacity would
cross the replacement threshold, one pack replacement is booked and spread over lifetime km.

In [3]:
r, h = cost_breakdown(dubai)
labels = {"vehicle_capex": "Vehicle capex", "energy": "Energy",
          "battery_degradation": "Battery reserve", "maintenance": "Maintenance",
          "insurance": "Insurance", "fleet_ops": "Fleet ops",
          "infrastructure": "Infrastructure", "driver_wages": "Driver wages"}
walk = pd.DataFrame({"Robotaxi ($/km)": [r[k] for k in labels],
                     "Human-driven ($/km)": [h[k] for k in labels]},
                    index=[labels[k] for k in labels])
print(walk.round(3).to_string())
print(f"\nDubai robotaxi total: ${r['total']:.3f}/km")
print(f"Dubai human-driven total: ${h['total']:.3f}/km")
print(f"Driver wages alone = ${h['driver_wages']:.3f}/km "
      f"({h['driver_wages']/h['total']:.0%} of the human-driven TCO)")

                 Robotaxi ($/km)  Human-driven ($/km)
Vehicle capex              0.089                0.060
Energy                     0.018                0.018
Battery reserve            0.026                0.029
Maintenance                0.040                0.040
Insurance                  0.013                0.015
Fleet ops                  0.078                0.000
Infrastructure             0.012                0.000
Driver wages               0.000                0.514

Dubai robotaxi total: $0.276/km
Dubai human-driven total: $0.676/km
Driver wages alone = $0.514/km (76% of the human-driven TCO)


## 3. Dubai vs Singapore comparison

In [4]:
dub_r, dub_h = cost_breakdown(dubai)
sg_r, sg_h = cost_breakdown(sg)
order = list(labels)
cats = [labels[k] for k in order]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, (p, title) in zip(axes, [(dubai, "Dubai"), (sg, "Singapore")]):
    rr, hh = cost_breakdown(p)
    bottom = np.zeros(2)
    for k in order:
        vals = np.array([rr[k], hh[k]])
        ax.bar(["Robotaxi", "Human-driven"], vals, bottom=bottom, label=labels[k])
        bottom += vals
    ax.set_title(f"{title} — TCO/km cost breakdown")
    ax.set_ylabel("$/km")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig("../assets/cost_breakdown.png", bbox_inches="tight")
plt.close(fig)

head = pd.DataFrame(
    {"Dubai": [dub_r["total"], dub_h["total"]],
     "Singapore": [sg_r["total"], sg_h["total"]]},
    index=["Robotaxi ($/km)", "Human-driven ($/km)"])
print(head.round(3).to_string())
for name, rr, hh in [("Dubai", dub_r, dub_h), ("Singapore", sg_r, sg_h)]:
    adv = hh["total"] - rr["total"]
    print(f"{name}: robotaxi advantage = ${adv:.3f}/km "
          f"({adv/hh['total']:.0%} cheaper than human-driven)")

                     Dubai  Singapore
Robotaxi ($/km)      0.276      0.501
Human-driven ($/km)  0.676      1.489
Dubai: robotaxi advantage = $0.400/km (59% cheaper than human-driven)
Singapore: robotaxi advantage = $0.987/km (66% cheaper than human-driven)


## 4. Breakeven analysis
Sweep utilization from 50 to 600 km/day for both fleets. The second question is more
revealing: how far would driver wages have to fall before human-driven wins?

In [5]:
ks = np.arange(50, 601, 25)
fig, ax = plt.subplots(figsize=(9, 4.6))
for p, name in [(dubai, "Dubai"), (sg, "Singapore")]:
    rk = [_robotaxi_total_at_util(p, k) for k in ks]
    hk = [_human_total_at_util(p, k) for k in ks]
    ax.plot(ks, rk, label=f"{name} robotaxi")
    ax.plot(ks, hk, label=f"{name} human-driven", linestyle="--")
ax.set_xlabel("Utilization (km/day)")
ax.set_ylabel("TCO ($/km)")
ax.set_title("TCO/km vs utilization — robotaxi cheaper across the full range")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("../assets/breakeven.png", bbox_inches="tight")
plt.close(fig)

bw_dubai = breakeven_driver_wage(dubai)
bw_sg = breakeven_driver_wage(sg)
print(f"Utilization breakeven: {breakeven_utilization(dubai)} "
      "(robotaxi already cheaper at the 50 km/day floor — no crossing in range)")
print(f"Driver-wage breakeven (Dubai): wages would need to fall to ${bw_dubai:.2f}/hr")
print(f"Driver-wage breakeven (Singapore): wages would need to fall to ${bw_sg:.2f}/hr")

Utilization breakeven: 50.0 (robotaxi already cheaper at the 50 km/day floor — no crossing in range)
Driver-wage breakeven (Dubai): wages would need to fall to $2.00/hr
Driver-wage breakeven (Singapore): wages would need to fall to $2.42/hr


## 5. Sensitivity analysis
Each key input varied ±30%; tornado shows the effect on the robotaxi cost advantage
(human TCO/km minus robotaxi TCO/km).

In [6]:
rows = sensitivity(dubai)
s_labels = [x["variable"] for x in rows]
low = [x["low"] - x["base"] for x in rows]
high = [x["high"] - x["base"] for x in rows]
y = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(y, low, color="#d62728", label="Input -30%")
ax.barh(y, high, color="#2ca02c", label="Input +30%")
ax.set_yticks(y); ax.set_yticklabels(s_labels)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Change in robotaxi cost advantage ($/km)")
ax.set_title("Dubai — sensitivity of the robotaxi advantage to ±30% input moves")
ax.legend(); fig.tight_layout()
fig.savefig("../assets/sensitivity.png", bbox_inches="tight")
plt.close(fig)
pd.DataFrame(rows)[["variable", "low", "high", "base"]].round(3)

,variable,low,high,base
0,Driver wage ($/hr),0.246,0.554,0.4
1,Robotaxi utilization (km/day),0.319,0.444,0.4
2,Fleet ops ($/veh/day),0.423,0.377,0.4
3,Autonomy stack premium ($),0.411,0.389,0.4
4,Infrastructure capex ($),0.404,0.396,0.4
5,Battery pack cost ($),0.399,0.401,0.4
6,Energy cost ($/kWh),0.400,0.400,0.4
7,Maintenance ($/km),0.400,0.400,0.4


## 6. Strategic & Commercial Implications for OEMs / Mobility Operators
Takeaways below are computed from this notebook's own model outputs (illustrative assumptions).

In [7]:
dub_r, dub_h = cost_breakdown(dubai)
sg_r, sg_h = cost_breakdown(sg)
adv_d = dub_h["total"] - dub_r["total"]
adv_s = sg_h["total"] - sg_r["total"]
top = sensitivity(dubai)[0]
batt_share = dub_r["battery_degradation"] / dub_r["total"]
infra_share = dub_r["infrastructure"] / dub_r["total"]
# what if depot/charging infra were shared and halved?
q = MarketParams(**{**dubai.__dict__,
                    "infra_capex_per_vehicle": dubai.infra_capex_per_vehicle / 2})
infra_saving = dub_r["infrastructure"] - cost_breakdown(q)[0]["infrastructure"]

print(f"1. The robotaxi business case is a labor-arbitrage case. In Dubai the model gives "
      f"robotaxi ${dub_r['total']:.2f}/km vs human-driven ${dub_h['total']:.2f}/km "
      f"({adv_d/dub_h['total']:.0%} cheaper); in Singapore ${sg_r['total']:.2f}/km vs "
      f"${sg_h['total']:.2f}/km ({adv_s/sg_h['total']:.0%} cheaper). Driver wages are "
      f"${dub_h['driver_wages']:.2f}/km in Dubai and ${sg_h['driver_wages']:.2f}/km in Singapore "
      f"— the single largest line item by far.")
print(f"\n2. The advantage is structurally robust: driver wages would need to collapse to "
      f"~${breakeven_driver_wage(dubai):.2f}/hr (Dubai) / ~${breakeven_driver_wage(sg):.2f}/hr "
      f"(Singapore) before human-driven breaks even — below any plausible labor market. "
      f"Sensitivity analysis confirms it: '{top['variable']}' moves the advantage by "
      f"${top['swing']:.2f}/km, ~{top['swing']/adv_d:.0%} of the total Dubai advantage.")
print(f"\n3. Utilization is the second lever, not the first. The robotaxi fleet is cheaper at "
      f"every utilization from 50–600 km/day in both markets, because robotaxi daily fixed "
      f"costs (~${(dub_r['vehicle_capex']+dub_r['fleet_ops']+dub_r['infrastructure']+dub_r['insurance'])*dubai.km_per_day_robotaxi:.0f}/day) "
      f"sit well below the human daily wage bill "
      f"(${dubai.driver_wage_per_hour*dubai.service_hours_per_day:.0f}/day). For operators, "
      f"this means the deployment risk is execution (fleet uptime, remote-ops cost control), "
      f"not demand density.")
print(f"\n4. Battery degradation is second-order. The pack-replacement reserve is "
      f"${dub_r['battery_degradation']:.3f}/km — {batt_share:.0%} of robotaxi TCO. "
      f"For OEMs this de-risks the durability debate: even doubling pack cost moves TCO "
      f"by only a few cents, while utilization and autonomy-stack cost dominate.")
print(f"\n5. Infrastructure sharing barely moves the math — halving depot/charging capex "
      f"saves ${infra_saving:.3f}/km (infrastructure is {infra_share:.0%} of TCO). "
      f"Shared charging hubs are nice for capex efficiency but are not the unlock; "
      f"autonomy-stack cost (${dubai.autonomy_stack_premium:,.0f}/vehicle) and fleet-ops "
      f"discipline are.")
print(f"\n6. Market selection matters more than technology cost curves. Singapore's COE "
      f"premium inflates vehicle capex for both fleets, yet the robotaxi advantage is "
      f"larger there (${adv_s:.2f}/km vs ${adv_d:.2f}/km) because the wage arbitrage is "
      f"bigger. Operators should rank launch markets by driver-wage levels and utilization "
      f"potential first, energy tariffs second.")

1. The robotaxi business case is a labor-arbitrage case. In Dubai the model gives robotaxi $0.28/km vs human-driven $0.68/km (59% cheaper); in Singapore $0.50/km vs $1.49/km (66% cheaper). Driver wages are $0.51/km in Dubai and $1.16/km in Singapore — the single largest line item by far.

2. The advantage is structurally robust: driver wages would need to collapse to ~$2.00/hr (Dubai) / ~$2.42/hr (Singapore) before human-driven breaks even — below any plausible labor market. Sensitivity analysis confirms it: 'Driver wage ($/hr)' moves the advantage by $0.31/km, ~77% of the total Dubai advantage.

3. Utilization is the second lever, not the first. The robotaxi fleet is cheaper at every utilization from 50–600 km/day in both markets, because robotaxi daily fixed costs (~$62/day) sit well below the human daily wage bill ($144/day). For operators, this means the deployment risk is execution (fleet uptime, remote-ops cost control), not demand density.

4. Battery degradation is second-order